In [1]:
import sys, os

sys.path.append(os.path.abspath(".."))

from src.spanish_only import *

device = 'cpu'
source = 'validation'

/home/david/Documents/Master/TFM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelos probados:
- paraphrase-multilingual-mpnet-base-v2
- paraphrase-multilingual-MiniLM-L12-v2

He visto que algunas propuestas interesantes involucran finetunning contrastivo o combinación de resultados de varios modelos sin entrenamiento. Se me ocurre que igual puedo plantear una mezcla de ambos enfoques, haciendo el finetunning unicamente en español y esperando que esto mejore los resultados.


# Basic evaluation

In [ ]:
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
nickname = 'mpnet-base'

In [15]:
run_spanish_evaluation(model_name, nickname, device, source)

Loading Spanish data...
Encoding data: paraphrase-multilingual-mpnet-base-v2 on device: cpu


Batches: 100%|██████████| 146/146 [02:17<00:00,  1.06it/s]


Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...
Spanish evaluation completed
Saving Spanish evaluation results...
Saved Spanish results to /home/david/Documents/Master/TFM/src/output/2025-11-19/005/results_spanish_monolingual.json
Updating Spanish ranking file...
Ranking español actualizado en /home/david/Documents/Master/TFM/src/output/ranking_spanish_validation.csv

Evaluación completada para paraphrase-multilingual-mpnet-base-v2
MAP español-español: 0.4167




# Training

## Preparacion de los datos

In [2]:
import pandas as pd
import itertools
import re
import unicodedata
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

In [3]:
train_df = pd.read_csv('../data/training/spanish/taskA_training_es.tsv', sep='\t', header=None)
train_df.columns = ['family_id', 'id', 'jobtitle_1', 'jobtitle_2']

train_df[['jobtitle_1', 'jobtitle_2']]

,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del Ejército del Aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
20719,encargado de vestuarios,encargada de vestidores
20720,encargado de vestuarios/encargada de vestuarios,encargada de vestuarios
20721,acomodadora,acomodador/acomodadora
20722,acomodador,acomodador/acomodadora


In [4]:
# dividir las palabras con marcadores de genero, separados por /

expanded = []

for _, row in train_df.iterrows():
    jt1 = row['jobtitle_1']
    jt2 = row['jobtitle_2']
    
    # Obtener opciones para cada columna
    opts1 = [x.strip() for x in jt1.split('/')]
    opts2 = [x.strip() for x in jt2.split('/')]
    
    # Crear combinaciones A × B
    for a, b in itertools.product(opts1, opts2):
        if a != b:
            expanded.append({
                'jobtitle_1': a,
                'jobtitle_2': b
            })

df_expanded = pd.DataFrame(expanded)
df_expanded


,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del Ejército del Aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
23868,encargado de vestuarios,encargada de vestidores
23869,encargado de vestuarios,encargada de vestuarios
23870,acomodadora,acomodador
23871,acomodador,acomodadora


In [5]:
# quedarnos solo con registros unicos, descartando los repetidos incluso cuando aparecen en la columna 1 y 2 intercambiados (ver ultimos tres registros del df_expanded)

df_temp = df_expanded.copy()

# Ordenar internamente cada pareja jobtitle_1 / jobtitle_2
df_temp[['jt1_sorted','jt2_sorted']] = (
    pd.DataFrame(
        df_temp.apply(lambda row: sorted([row['jobtitle_1'].strip(),
                                          row['jobtitle_2'].strip()]),
                      axis=1).to_list(),
        index=df_temp.index
    )
)

df_unique = df_temp.drop_duplicates(subset=['jt1_sorted','jt2_sorted'])
df_unique = df_unique[['jobtitle_1', 'jobtitle_2']].reset_index(drop=True)

df_unique

,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del Ejército del Aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
17882,encargada de vestidores,encargada de vestuarios
17883,encargado de vestuarios,encargada de vestuarios
17884,encargado de vestuarios,encargado de vestidores
17885,encargada de vestidores,encargado de vestuarios


In [6]:
def normalize_titles(title):

    if pd.isna(title):
        return ""

    title = str(title).lower().strip()
    title = unicodedata.normalize('NFC', title)
    title = re.sub(r'[^\w\s]', '', title)   # creo que no esta funcionando correctamente. Revisar funciones de trabajos del master
    title = unicodedata.normalize('NFC', title)
    title = re.sub(r'\s+', ' ', title)
    title = title.strip('.,;:!?-')

    return title.strip()


def normalize_dataframe(df):

    df_normalized = df.copy()
    columns = df.columns.values
    
    for col in columns:
        if col in df_normalized.columns:
            df_normalized[col] = df_normalized[col].apply(normalize_titles)
    
    return df_normalized



df_normalized = normalize_dataframe(df_unique)
df_normalized

,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del ejército del aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
17882,encargada de vestidores,encargada de vestuarios
17883,encargado de vestuarios,encargada de vestuarios
17884,encargado de vestuarios,encargado de vestidores
17885,encargada de vestidores,encargado de vestuarios


In [7]:
train_examples = []

for idx, row in df_normalized.iterrows():
    example = InputExample(texts=[row['jobtitle_1'], row['jobtitle_2']])
    train_examples.append(example)

print(f"\nCreados {len(train_examples)} ejemplos de entrenamiento")


Creados 17887 ejemplos de entrenamiento


## Entrenamiento del modelo

In [8]:
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(model_name, device=device)

In [9]:
output_path = './finetune/first-attempt'
epochs = 1
warmup_steps = 100
batch_size = 8

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

# Loss function: InfoNCE (aprendizaje contrastivo)
# Los negativos se generan automáticamente del mismo batch
train_loss = losses.MultipleNegativesRankingLoss(model)

In [11]:
import torch

os.environ['CUDA_VISIBLE_DEVICES'] = ''
torch.cuda.is_available = lambda : False

# Verificar que NO hay GPU disponible
print(f"CUDA disponible: {torch.cuda.is_available()}")

CUDA disponible: False


In [12]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    output_path=output_path,
    show_progress_bar=True,
    use_amp=False
)

/home/david/Documents/Master/TFM/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


KeyboardInterrupt: 